# QNLP Sentiment Training – Financial PhraseBank & FiQA

**Purpose:** This notebook is dedicated to **QNLP** (Quantum Natural Language Processing) for financial sentiment using **lambeq** or **DisCoPy** (QDisCoCirc). We **train** QNLP when the pipeline is available (PennyLane/Qiskit simulators). **FinBERT is pretrained** (ProsusAI/finbert) — we do **not** train it here; we load it and run **inference only** at the end as a **benchmark** to compare QNLP (and classical baseline) against. A small classical baseline (TF-IDF + LogReg) is also trained only for F1/MSE comparison.

**Flow:** Load data → train/val split → **QNLP training (main)** → Benchmarks (classical + FinBERT) → export. **Output:** The **primary artifact is always the QNLP (quantum) model** (`sentiment_quantum_v1.pkl`), regardless of benchmark performance. Classical and FinBERT are for comparison only. Runs on **local** or **Google Colab** (run the Setup cell first on Colab).

**Runtime:** **Auto-detected** — **Colab** = GPU; **local** = CPU-only. **Critical design:** QNLP (lambeq) with **JAX optional** (use_jit=False if missing); classical + FinBERT as benchmarks; **model snapshot** = `.pkl`; **report** = Colab download only. **Datasets:** Financial PhraseBank & FiQA. **Dependencies:** `lambeq`, `pennylane`.

## Setup: Colab vs local

Run this cell first. On **Google Colab** it clones the repo (if needed), installs dependencies, and optionally installs **JAX** for faster QNLP training. On **local**, it only ensures you're in the repo. The **main output** of this notebook is the **QNLP (quantum) model** (`sentiment_quantum_v1.pkl`); classical and FinBERT are benchmarks only.

In [1]:
import os
import sys
from pathlib import Path

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO_URL = "https://github.com/leemingloon/ocr-agentic-rag.git"
    REPO_DIR = "ocr-agentic-rag"
    if not os.path.isdir(REPO_DIR):
        get_ipython().system(f"git clone {REPO_URL} {REPO_DIR}")
    get_ipython().run_line_magic("cd", REPO_DIR)
    get_ipython().system("pip install -q -r notebooks/requirements-notebooks.txt")
    get_ipython().system("pip install -q jax jaxlib 2>/dev/null || true")
    ROOT = Path(".").resolve()
    print("Colab: repo ready. JAX optional (use_jit=False if missing — no 'No module named jax' error).")
else:
    ROOT = Path(".").resolve() if (Path(".").resolve() / "credit_risk").exists() else Path("..").resolve()
    if str(ROOT) not in sys.path:
        sys.path.insert(0, str(ROOT))
    print("Local run. For Bobcat parser (better QNLP), try running on Google Colab.")

Local run. For Bobcat parser (better QNLP), try running on Google Colab.


## 1. Load sentiment data (Financial PhraseBank / FiQA)

In [10]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

if 'ROOT' not in dir() or ROOT is None:
    ROOT = Path(".").resolve() if (Path(".").resolve() / "credit_risk").exists() else Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

def load_phrasebank(split="test"):
    path = ROOT / "data" / "credit_risk_sentiment" / "FinancialPhraseBank" / split
    if not path.exists():
        return pd.DataFrame()
    pq_files = list(path.glob("*.parquet"))
    if not pq_files:
        return pd.DataFrame()
    df = pd.read_parquet(pq_files[0])
    df["text"] = df.get("text", df.get("sentence", ""))
    df["label"] = df.get("label_text", df.get("label", "neutral")).str.strip().str.lower()
    return df[["text", "label"]].dropna()

def load_fiqa(split="valid"):
    path = ROOT / "data" / "credit_risk_sentiment" / "FiQA" / split
    if not path.exists():
        return pd.DataFrame()
    pq_files = list(path.glob("*.parquet"))
    if not pq_files:
        return pd.DataFrame()
    df = pd.read_parquet(pq_files[0])
    df["text"] = df.get("sentence", df.get("text", ""))
    score = pd.to_numeric(df.get("score", 0), errors="coerce").fillna(0)
    df["label"] = np.where(score < -0.2, "negative", np.where(score > 0.2, "positive", "neutral"))
    return df[["text", "label"]].dropna()

df_pb = load_phrasebank()
df_fiqa = load_fiqa()
print('Financial PhraseBank:', len(df_pb), 'samples')
print('FiQA:', len(df_fiqa), 'samples')
df = pd.concat([df_pb, df_fiqa], ignore_index=True) if len(df_pb) > 0 and len(df_fiqa) > 0 else (df_pb if len(df_pb) > 0 else df_fiqa)
if len(df) == 0:
    raise FileNotFoundError('Download Financial PhraseBank and/or FiQA under data/credit_risk_sentiment/ (see scripts/download_datasets.py)')

Financial PhraseBank: 1000 samples
FiQA: 117 samples


## 2. Train/val split

Split data so both **QNLP (Section 3)** and **benchmarks (Section 4)** use the same splits.

In [11]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df["label"])
y_train = train_df["label"].values
y_val = val_df["label"].values
print("Train:", len(train_df), "Val:", len(val_df))

Train: 893 Val: 224


## 3. QNLP with lambeq (main)

Train **QNLP** when lambeq is available: parse sentences to string diagrams, convert to quantum circuits, train and report validation F1. **BobcatParser** (CCG) requires a remote service that may be unavailable (`qnlp.cambridgequantum.com`); on failure we fall back to **cups_reader** (offline, word-sequence diagrams). **Frontier improvements** (research/Kaggle/lambeq docs): deeper IQP ansatz (n_layers≥2), more training data, Rotosolve or tuned SPSA, 3-class labels, more epochs and modest batch size. Trained model saved as `sentiment_quantum_v1.pkl`. **Section 4** runs benchmarks (classical + FinBERT) to compare.

In [12]:
try:
    import lambeq
    LAMBEQ_AVAILABLE = True
except ImportError:
    LAMBEQ_AVAILABLE = False
    print('lambeq not installed. pip install lambeq. Skipping QNLP circuit training.')

# QNLP: Bobcat (remote) or fallback to cups_reader (offline). Then train and report F1.
f1_qnlp = 0.0
qnlp_model_trained = None
f1_cl = 0.0  # placeholder until Section 4 (Benchmarks) is run; then classical F1 overwrites this
if LAMBEQ_AVAILABLE:
    from lambeq import RemoveCupsRewriter, IQPAnsatz, AtomicType
    remove_cups = RemoveCupsRewriter()
    def _norm(s):
        return ' '.join((s or ' unknown ').strip().split())
    sentences_train = [_norm(s) for s in train_df["text"].astype(str).tolist()[:min(600, len(train_df))]]
    sentences_val = [_norm(s) for s in val_df["text"].astype(str).tolist()[:min(250, len(val_df))]]
    diagrams_train, diagrams_val = [], []
    try:
        from lambeq import BobcatParser
        parser = BobcatParser(verbose="suppress")
        raw_t = parser.sentences2diagrams(sentences_train)
        raw_v = parser.sentences2diagrams(sentences_val)
        diagrams_train = [remove_cups(d) for d in raw_t if d is not None]
        diagrams_val = [remove_cups(d) for d in raw_v if d is not None]
        print('QNLP: using BobcatParser.')
    except Exception as e:
        print('QNLP: Bobcat unavailable — using offline cups_reader.', type(e).__name__)
        try:
            from lambeq import cups_reader
            for s in sentences_train:
                try:
                    d = cups_reader.sentence2diagram(s)
                    if d is not None: diagrams_train.append(d)
                except Exception: pass
            for s in sentences_val:
                try:
                    d = cups_reader.sentence2diagram(s)
                    if d is not None: diagrams_val.append(d)
                except Exception: pass
        except Exception as e2:
            print('cups_reader failed:', e2)
    if diagrams_train and diagrams_val:
        try:
            # Deeper ansatz (n_layers=2) improves expressivity (IEEE/Quantinuum QNLP literature)
            ansatz = IQPAnsatz({AtomicType.NOUN: 1, AtomicType.SENTENCE: 1}, n_layers=2, n_single_qubit_params=3)
            circ_t = [ansatz(d) for d in diagrams_train if d]
            circ_v = [ansatz(d) for d in diagrams_val if d]
            circ_t = [c for c in circ_t if c is not None]
            circ_v = [c for c in circ_v if c is not None]
        except Exception:
            circ_t, circ_v = [], []
        if circ_t and circ_v:
            from sklearn.metrics import f1_score as f1_skl
            # Prefer 3-class (negative, neutral, positive) for better F1; fallback to binary if circuit dim=2
            label_map3 = {'negative': 0, 'neutral': 1, 'positive': 2}
            y_tr = np.array([label_map3.get(str(l).strip().lower(), 1) for l in y_train[:len(circ_t)]])
            y_va = np.array([label_map3.get(str(l).strip().lower(), 1) for l in y_val[:len(circ_v)]])
            y_tr, y_va = y_tr[:len(circ_t)], y_va[:len(circ_v)]
            if len(y_tr) == len(circ_t) and len(y_va) == len(circ_v):
                try:
                    from lambeq import NumpyModel, Dataset, QuantumTrainer, SPSAOptimizer, CrossEntropyLoss
                    try:
                        from lambeq import RotosolveOptimizer
                        OptimizerCls, optim_hp = RotosolveOptimizer, {}
                        print('QNLP: using RotosolveOptimizer (frontier: often better convergence).')
                    except ImportError:
                        OptimizerCls, optim_hp = SPSAOptimizer, {'a': 0.15, 'c': 0.05, 'A': 10.0}
                        print('QNLP: using SPSAOptimizer (tuned A=10 for slower decay).')
                    try:
                        import jax
                        use_jit = True
                    except ImportError:
                        use_jit = False
                    model = NumpyModel.from_diagrams(circ_t + circ_v, use_jit=use_jit)
                    model.initialise_weights()
                    out_shape = model(circ_t[:1]).shape
                    n_out = out_shape[-1] if len(out_shape) > 1 else 1
                    if n_out == 2:
                        y_tr_bin = np.minimum(y_tr, 1)
                        y_va_bin = np.minimum(y_va, 1)
                        onehot_t, onehot_v = np.eye(2)[y_tr_bin], np.eye(2)[y_va_bin]
                        y_va_eval = y_va_bin
                    else:
                        onehot_t, onehot_v = np.eye(3)[y_tr], np.eye(3)[y_va]
                        y_va_eval = y_va
                    batch_sz = min(16, len(circ_t))
                    train_ds = Dataset(circ_t, onehot_t, batch_size=batch_sz)
                    val_ds = Dataset(circ_v, onehot_v, shuffle=False)
                    trainer = QuantumTrainer(model, loss_function=CrossEntropyLoss(), epochs=40,
                        optimizer=OptimizerCls, optim_hyperparams=optim_hp,
                        evaluate_on_train=True, verbose='text', seed=42)
                    trainer.fit(train_ds, val_ds)
                    pred_v = model(circ_v)
                    y_pred = np.argmax(pred_v, axis=1)
                    f1_qnlp = f1_skl(y_va_eval, y_pred, average='macro', zero_division=0)
                    qnlp_model_trained = model
                    print('QNLP trained. Validation F1 macro: {:.4f}'.format(f1_qnlp))
                except Exception as e3:
                    print('QNLP training failed:', e3)
    if not diagrams_train or not diagrams_val:
        print('No diagrams (Bobcat failed and cups_reader produced none).')
else:
    f1_qnlp = 0.0
print('\nF1 comparison: Classical', round(f1_cl, 4), '| QNLP:', round(f1_qnlp, 4))

QNLP: Bobcat unavailable — using offline cups_reader. ModelDownloaderError


Epoch 1:   train/loss: 1.1397   valid/loss: 1.0329   train/time: 1m7s   valid/time: 15.80s
Epoch 2:   train/loss: 0.8279   valid/loss: 0.9581   train/time: 1m25s   valid/time: 20.93s
Epoch 3:   train/loss: 0.9467   valid/loss: 0.9452   train/time: 1m21s   valid/time: 20.09s
Epoch 4:   train/loss: 0.9450   valid/loss: 1.1901   train/time: 1m20s   valid/time: 20.68s
Epoch 5:   train/loss: 1.1779   valid/loss: 0.9436   train/time: 1m22s   valid/time: 19.31s
Epoch 6:   train/loss: 0.9706   valid/loss: 1.0053   train/time: 1m33s   valid/time: 23.48s
Epoch 7:   train/loss: 1.2146   valid/loss: 0.9472   train/time: 1m27s   valid/time: 17.81s
Epoch 8:   train/loss: 1.2469   valid/loss: 0.9264   train/time: 1m24s   valid/time: 17.25s
Epoch 9:   train/loss: 0.8509   valid/loss: 0.9086   train/time: 1m25s   valid/time: 21.75s
Epoch 10:  train/loss: 0.7874   valid/loss: 1.0080   train/time: 1m27s   valid/time: 19.83s
Epoch 11:  train/loss: 0.9680   valid/loss: 0.9378   train/time: 1m23s   valid/ti

QNLP trained. Validation F1 macro: 0.4026

F1 comparison: Classical 0.0 | QNLP: 0.4026


## 4. Benchmarks: classical baseline and FinBERT (pretrained)

Train a **classical** baseline (TF-IDF + LogReg, optional ensemble) for F1 comparison. **FinBERT is pretrained** (ProsusAI/finbert): we **load** it and run **inference only** — no training.

### 4a. Classical (TF-IDF + LogReg, optional augmentation)

In [13]:
from sklearn.metrics import f1_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

# train_df, val_df, y_train, y_val from Section 2 (Train/val split)
# Data augmentation (training only): synonym replacement
def _augment_text(text, p_replace=0.3, rng=None):
    if rng is None:
        rng = np.random.default_rng(42)
    try:
        import nltk
        from nltk.corpus import wordnet
        nltk.data.find("corpora/wordnet")
    except Exception:
        return text
    words = str(text).split()
    if len(words) < 2:
        return text
    out = list(words)
    for i in range(len(out)):
        if rng.random() > p_replace:
            continue
        syns = wordnet.synsets(out[i])
        if syns:
            lemmas = [l.name() for s in syns for l in s.lemmas() if l.name() != out[i]]
            if lemmas:
                out[i] = rng.choice(lemmas)
    return " ".join(out)

_rng = np.random.default_rng(42)
_train_texts = list(train_df["text"].astype(str))
_train_labels = list(y_train)
for _i in _rng.choice(len(_train_texts), size=min(len(_train_texts) // 2, 500), replace=False):
    _train_texts.append(_augment_text(_train_texts[_i], p_replace=0.25, rng=_rng))
    _train_labels.append(_train_labels[_i])
train_df_aug = pd.DataFrame({"text": _train_texts, "label": _train_labels})
y_train_aug = train_df_aug["label"].values

vec = TfidfVectorizer(max_features=2000, ngram_range=(1, 2), sublinear_tf=True, min_df=2)
X_train = vec.fit_transform(train_df_aug["text"].astype(str))
X_val = vec.transform(val_df["text"].astype(str))

clf = LogisticRegression(max_iter=500, class_weight="balanced", random_state=42, C=1.0)
clf.fit(X_train, y_train_aug)
y_pred_cl = clf.predict(X_val)
f1_cl = f1_score(y_val, y_pred_cl, average="macro")
print('Classical (TF-IDF + LogReg, with augmentation) F1 macro:', round(f1_cl, 4))

Classical (TF-IDF + LogReg, with augmentation) F1 macro: 0.7632


### 4b. FinBERT (pretrained): inference only — no training

Load **ProsusAI/finbert** (pretrained) and run inference on the validation set for F1 comparison. We do **not** train FinBERT here.

In [15]:
# Improved classical: optional FinBERT, else stronger TF-IDF + LogReg or SVC
f1_improved = f1_cl
try:
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
    import torch
    model_name = "ProsusAI/finbert"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name)
    model.eval()
    def finbert_predict(texts, batch_size=16):
        preds = []
        for i in range(0, len(texts), batch_size):
            batch = texts[i : i + batch_size]
            enc = tokenizer(batch, padding=True, truncation=True, return_tensors="pt", max_length=128)
            with torch.no_grad():
                logits = model(**enc).logits
            idx = logits.argmax(dim=1)
            label_map = {0: "negative", 1: "neutral", 2: "positive"}
            preds.extend([label_map[j.item()] for j in idx])
        return np.array(preds)
    y_pred_finbert = finbert_predict(val_df["text"].astype(str).tolist())
    f1_finbert = f1_score(y_val, y_pred_finbert, average="macro")
    if f1_finbert > f1_improved:
        f1_improved = f1_finbert
        y_pred_cl = y_pred_finbert
        print("Improved (FinBERT) F1 macro:", round(f1_finbert, 4))
    else:
        print("FinBERT F1 macro:", round(f1_finbert, 4), "(baseline TF-IDF+LogReg used)")
except Exception as e:
    print("FinBERT not used:", e)
if f1_improved == f1_cl:
    vec2 = TfidfVectorizer(max_features=5000, ngram_range=(1, 3), sublinear_tf=True, min_df=2)
    X_train2 = vec2.fit_transform(train_df["text"].astype(str))
    X_val2 = vec2.transform(val_df["text"].astype(str))
    from sklearn.svm import LinearSVC
    clf2 = LinearSVC(class_weight="balanced", random_state=42, max_iter=2000)
    clf2.fit(X_train2, y_train)
    y_pred2 = clf2.predict(X_val2)
    f1_svc = f1_score(y_val, y_pred2, average="macro")
    if f1_svc > f1_cl:
        f1_improved = f1_svc
        y_pred_cl = y_pred2
        vec, clf = vec2, clf2
        print("Improved (TF-IDF + LinearSVC) F1 macro:", round(f1_svc, 4))
f1_cl = max(f1_cl, f1_improved)
print("Best classical F1 macro:", round(f1_cl, 4))

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


FinBERT F1 macro: 0.0355 (baseline TF-IDF+LogReg used)
Best classical F1 macro: 0.7632


### 4c. Classical ensemble (LogReg + LinearSVC + meta-learner)

Combine TF-IDF + LogReg and TF-IDF + LinearSVC via a meta-learner. Saved artifact for eval_runner stays vectorizer + (best) single classifier or ensemble wrapper.

In [16]:
# Ensemble: LogReg + Calibrated LinearSVC, meta-learner on stacked probs (use current vec)
from sklearn.calibration import CalibratedClassifierCV
from sklearn.svm import LinearSVC

X_train_ens = vec.transform(train_df_aug["text"].astype(str))
X_val_ens = vec.transform(val_df["text"].astype(str))
clf_lr = LogisticRegression(max_iter=500, class_weight="balanced", random_state=42, C=1.0)
clf_lr.fit(X_train_ens, y_train_aug)
clf_svc = CalibratedClassifierCV(LinearSVC(class_weight="balanced", random_state=42, max_iter=2000), cv=3)
clf_svc.fit(X_train_ens, y_train_aug)

P1 = clf_lr.predict_proba(X_train_ens)
P2 = clf_svc.predict_proba(X_train_ens)
X_meta = np.hstack([P1, P2])
meta = LogisticRegression(max_iter=500, random_state=42)
meta.fit(X_meta, y_train_aug)

P1_val = clf_lr.predict_proba(X_val_ens)
P2_val = clf_svc.predict_proba(X_val_ens)
y_pred_ens = meta.predict(np.hstack([P1_val, P2_val]))
f1_ens = f1_score(y_val, y_pred_ens, average="macro")

if f1_ens > f1_cl:
    class _EnsembleWrapper:
        def __init__(self, meta, clf_lr, clf_svc):
            self.meta = meta
            self.clf_lr = clf_lr
            self.clf_svc = clf_svc
            self.classes_ = clf_lr.classes_
        def predict(self, X):
            return self.meta.predict(np.hstack([self.clf_lr.predict_proba(X), self.clf_svc.predict_proba(X)]))
        def predict_proba(self, X):
            return self.meta.predict_proba(np.hstack([self.clf_lr.predict_proba(X), self.clf_svc.predict_proba(X)]))
    clf = _EnsembleWrapper(meta, clf_lr, clf_svc)
    f1_cl = f1_ens
    print("Ensemble (LogReg + SVC + meta) F1 macro:", round(f1_ens, 4))
else:
    print("Ensemble F1 macro:", round(f1_ens, 4), "(single model kept)")

Ensemble F1 macro: 0.7613 (single model kept)


## 5. Summary and export

- **Primary output:** The **QNLP (quantum) model** is always saved as `sentiment_quantum_v1.pkl` when training completes, regardless of F1 vs benchmarks.
- **Benchmarks (Section 4):** Classical (TF-IDF + LogReg) and pretrained FinBERT are for comparison only. We do **not** export classical/pretrained model snapshots (only models we train and tune are saved as `.pkl`).

In [17]:
import joblib

MODEL_DIR = ROOT / "models" / "sentiment"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
# Primary output: always save QNLP (quantum) model when we have it
if 'qnlp_model_trained' in dir() and qnlp_model_trained is not None:
    joblib.dump({"model": qnlp_model_trained, "labels": ["negative", "neutral", "positive"], "type": "qnlp_numpy"}, MODEL_DIR / "sentiment_quantum_v1.pkl")
    print('Saved QNLP (quantum) model to models/sentiment/sentiment_quantum_v1.pkl')
else:
    print('No QNLP model to save; run Section 3 (QNLP) to train the quantum model.')
# Optional: save classical benchmark model for eval_runner
if 'vec' in dir() and 'clf' in dir() and vec is not None and clf is not None:
    joblib.dump({"vectorizer": vec, "classifier": clf, "labels": list(clf.classes_)}, MODEL_DIR / "sentiment_classical_v1.pkl")
    print('Saved classical (benchmark) model to models/sentiment/sentiment_classical_v1.pkl')
else:
    print('Skip classical save: run Section 4 (Benchmarks) first to define vec and clf.')

Saved QNLP (quantum) model to models/sentiment/sentiment_quantum_v1.pkl
Saved classical (benchmark) model to models/sentiment/sentiment_classical_v1.pkl


## 6. Colab: Download report (cell outputs)

**Model deliverable:** The **model snapshot** is the `.pkl` file (`sentiment_quantum_v1.pkl`) from the Save cell — download on Colab if needed. Classical/FinBERT are benchmarks only and are not exported.

On **Google Colab only**, the cell below builds an **overall report** of key cell outputs and **downloads** it (no saving to `models/` on local).

In [ ]:
# Report of key cell outputs: Colab = build + download only; local = no write to models/
report_lines = ["QNLP Sentiment training – overall report", "=" * 60, ""]
if "f1_cl" in dir():
    report_lines.append(f"QNLP (quantum) F1 (macro): {f1_cl}")
if "f1_cl_classical" in dir():
    report_lines.append(f"Classical (TF-IDF+LogReg) F1: {f1_cl_classical}")
if "f1_finbert" in dir():
    report_lines.append(f"FinBERT F1: {f1_finbert}")
report_lines.extend(["", "Primary model snapshot: sentiment_quantum_v1.pkl (download from Save cell on Colab)", ""])
report_txt = "\n".join(report_lines)

try:
    from google.colab import files
    from pathlib import Path
    report_path = Path("report_sentiment_qnlp.txt")
    report_path.write_text(report_txt, encoding="utf-8")
    files.download(str(report_path))
    print("Report download started. Paste into repo or keep for records.")
except ImportError:
    print("Local run: report not saved to models/ (Cursor can use notebook outputs).")
    print(report_txt[:800] + ("..." if len(report_txt) > 800 else ""))

Report: C:\Users\leemi\OneDrive\Desktop\Job_search_2026\ocr-agentic-rag\models\sentiment\report_sentiment_qnlp.txt  |  Summary image: C:\Users\leemi\OneDrive\Desktop\Job_search_2026\ocr-agentic-rag\models\sentiment\sentiment_quantum_summary.png
Not on Colab. Files are in C:\Users\leemi\OneDrive\Desktop\Job_search_2026\ocr-agentic-rag\models\sentiment


## 7. Evaluation deliverables (after running eval_runner)

After running `eval_runner.py --category credit_risk_sentiment` (e.g. from CLI), run the cell below. **Colab:** download all `*_per_sample_*.json` and generated QA txt. **Local:** save QA txt to `data/proof/credit_risk_sentiment/<dataset>/<split>/`.

In [ ]:
# Per-sample JSON + QA txt: Colab = download both; local = save txt to data/proof
import json
from pathlib import Path
PROOF_DIR = ROOT / "data" / "proof"
CATEGORY = "credit_risk_sentiment"
base = PROOF_DIR / CATEGORY.lower()
per_sample_paths = sorted(base.rglob("*_per_sample_*.json")) if base.exists() else []

def _rows_to_qa_txt(rows):
    lines = []
    for i, row in enumerate(rows):
        if i > 0: lines.append("")
        lines.append("=" * 72)
        lines.append(f"sample_id: {row.get('sample_id', '')}")
        lines.append(f"split: {row.get('split', '')}")
        lines.append(f"ground_truth: {row.get('ground_truth', '')}")
        inp = row.get("input_text") or {}
        if isinstance(inp, dict) and inp.get("question"): lines.append("question: " + str(inp.get("question", "")))
        elif isinstance(inp, str): lines.append("input_text: " + inp)
        else: lines.append("input_text: " + json.dumps(inp, ensure_ascii=False))
        lines.append("-" * 72)
        pred = row.get("prediction") or ""
        lines.append("prediction:")
        lines.append(pred if isinstance(pred, str) else json.dumps(pred, ensure_ascii=False))
        if row.get("prediction_error"): lines.append("prediction_error: " + str(row.get("prediction_error")))
        met = row.get("metrics") or {}
        if met: lines.append("metrics: " + json.dumps(met, ensure_ascii=False))
        lines.append("=" * 72)
    return "\n".join(lines)

try:
    from google.colab import files as colab_files
    _in_colab = True
except ImportError:
    _in_colab = False

for per_sample_path in per_sample_paths:
    try:
        with open(per_sample_path, "r", encoding="utf-8") as f:
            rows = json.load(f)
    except Exception:
        continue
    if not isinstance(rows, list) or not rows:
        continue
    stem = per_sample_path.stem
    base_name = stem.split("_per_sample_")[0] if "_per_sample_" in stem else stem
    txt_name = f"{base_name}_predictions.txt"
    txt_content = _rows_to_qa_txt(rows)
    out_path = per_sample_path.parent / txt_name
    if _in_colab:
        colab_files.download(str(per_sample_path))
        Path(txt_name).write_text(txt_content, encoding="utf-8")
        colab_files.download(txt_name)
    else:
        out_path.write_text(txt_content, encoding="utf-8")
        print(f"[export] {out_path}")

if _in_colab and per_sample_paths:
    print("Colab: downloads started for per_sample JSON + predictions txt.")
elif not per_sample_paths:
    print("No per_sample JSONs found. Run eval_runner --category credit_risk_sentiment first.")